<h1><center>Laboratorio 5: El Oso Polar no muerde 🐻‍❄️</center></h1>

<center><strong>IA7202: Laboratorio de Programación Científica para Ciencia de Datos</strong></center>

<div style="text-align: center; margin: 24px 0;">
  <img
    src="assets/polar.png"
    alt="We Bare Bears - Ice Bear With Axe"
    style="width: 400px; max-width: 100%; height: auto;"
  >
</div>

----

### Cuerpo Docente

- Profesores: Pablo Badilla e Ignacio Núñez
- Auxiliar: Sofía Chávez
- Ayudantes: Javiera Arévalo, Tamara Carrasco, Ignacio Reyes


### Equipo

Completen los nombres de los integrantes del equipo.

**Importante:** Los notebooks sin esta información no se revisarán.

- Nombre de estudiante 1:
- Nombre de estudiante 2:

### Link de la Pull Request

**Importante:** Peguen aquí la URL HTTPS completa de la Pull Request de su laboratorio:


`https://github.com/<organizacion>/<repositorio>/pull/<numero>`




---

### Reglas

- **Grupos de máximo 2 personas**.
- Las dudas fuera de horario se reciben en el foro del curso o por correo al equipo docente.


<!-- IMPORTANTE -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #8957e5; background:rgba(137,87,229,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">📌 Importante</strong>

  Está prohibido el uso de ciclos `for` ni `while` para recorrer u operar datos. Utilicen siempre los métodos nativos de `Polars` sobre columnas completas.

  Tampoco está permitido el uso de `pandas` ya que esta no es la tecnología trabajada en este laboratorio.
</div>

### Flujo de trabajo


En la mayoría de los ejercicios, primero deberán resolver uno o varios problemas acotados y, en una sección posterior, deberán convertir esa solución en módulos que después, podremos reutilizar e integrar en sistemas más complejos.

El laboratorio sigue una dinámica de desarrollo guiado por pruebas
(*Test-Driven Development*, **TDD**): cada etapa incluye pruebas que sus soluciones deben pasar.

Consideren una etapa terminada cuando pasen todas las pruebas asociadas.

### Configuración del Ambiente

Utilicen el siguiente comando desde la raíz del proyecto para instalar las librerías requeridas en el laboratorio:

```bash
uv add polars plotly pandas numpy scipy jupyterlab ipywidgets pyarrow
uv sync
```

Y asegúrense que eligen el ambiente virtual de su proyecto raíz usando las siguientes instrucciones:

1. Abrir la paleta de comandos con: `Control + Shift + P` o `F1`.
2. Buscar: `Python: Select Interpreter` o su traducción en español.
3. Seleccionar el ambiente virtual desde la carpeta base del repositorio del laboratorio (algo similar a `.venv/Scripts/python`, en dónde la carpeta base debería ser la raíz de su repositorio).


## Contexto

<!-- LORE -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #0f9e99; background:rgba(15,158,153,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">📜 El encargo del Director Polar</strong>

  <div style="text-align: center; margin: 24px 0;">
    <img
      src="assets/colegio.png"
      alt="We Bare Bears - Ice Bear With Axe"
      style="width: 600px; max-width: 100%; height: auto;"
    >
  </div>



  Polar, director del Colegio Patagónico del Pingüino Helado, necesita ordenar y analizar los registros de notas del colegio, una tarea importantísima para entender mejor el rendimiento y comportamiento de sus estudiantes.

  Para esto, decide crear una nueva subdirección de análisis de datos y ponerlos a ustedes a cargo de los primeros análisis.

  El sistema informático principal contiene 875 matrículas, repartidas en dos archivos JSON: `students_grades_1.json` y `students_grades_2.json`. Sin embargo, otro sistema del colegio entregó un CSV (`other_grades.csv`) con 1000 registros de notas adicionales.

  Como podrán imaginar, algo no cuadra. El director Polar sospecha que puede haber estudiantes ya graduados dentro del conjunto de notas, registros duplicados e incluso alumnos que derechamente no existen y corresponden a errores del sistema.

  Su primera tarea como flamante equipo de análisis será descubrir qué está pasando con estos datos.

</div>


El objetivo de este laboratorio es continuar mejorando las habilidades de manipulación de datos con `Polars`, a la vez que exploramos diversos gráficos con `Plotly`.


Los objetivos son:

- Reconstruir una fuente fragmentada y comparar esquemas;
- Auditar claves antes de unir tablas;
- Distinguir tablas anchas y largas;
- Calcular indicadores sin mezclar escalas;
- Construir visualizaciones adecuadas para cada pregunta;
- Trasladar las operaciones a funciones testeables y a un pipeline lazy.

### Evaluación

| Etapa | Actividades | Puntaje |
|---|---|---:|
| 1 | Lectura 0,2; exploración 0,2; interpretación 0,2; módulo 0,2 | 0,8 |
| 2 | Auditoría 0,2; seis joins 0,6; decisión 0,2; módulo 0,2 | 1,2 |
| 3 | Ejemplo 0,1; unpivot 0,2; escalas 0,2; unicidad 0,1; pivot 0,2; módulo 0,3 | 1,1 |
| 4 | GPA 0,2; mapeo 0,2; resumen 0,3; ranking 0,1; asignaturas 0,1; interpretación 0,2 | 1,1 |
| 5 | Cuatro gráficos base 0,4; dos extras 0,2; interpretación 0,2; módulo 0,2 | 1,0 |
| 6 | Pipeline 0,2; explain y materialización 0,2; exportación 0,2; módulo y entrega 0,2 | 0,8 |
| **Total** | | **6,0** |

La nota del laboratorio se calcula como 1,0 + puntaje obtenido.

---
## Etapa 1 — I/O y reconstrucción de fuentes (0,8 puntos)

### 1.1 — Preparar las rutas

<!-- IMPORTANTE -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #8957e5; background:rgba(137,87,229,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">📌 Importante</strong>

  Ejecuten siempre el notebook desde la carpeta <code>notebooks/</code>, ya que el laboratorio está preconfigurado para trabajar desde esa ubicación.

  La siguiente celda agrega la raíz del proyecto al entorno de ejecución para que los imports locales funcionen correctamente y deja definidas las rutas de entrada y salida que utilizaremos durante el laboratorio.

  <strong>No modifiquen esta celda.</strong>
</div>

#### Importamos las Librerías y Definimos las Rutas

In [ ]:
from pathlib import Path
import sys

import plotly.express as px
import polars as pl

# -----------------------------------------------------------------
# 1. Obtenemos la ruta actual con cwd (current working directory)
# y luego, buscamos la ruta para agregarla al directorio de trabajo
# del notebook.
RUTA_LAB = Path.cwd().parent
if str(RUTA_LAB) not in sys.path:
    sys.path.insert(0, str(RUTA_LAB))

# -----------------------------------------------------------------
# 2. Definimos las rutas de los datos de entrada y artefactos
# de salida, según las definiciones de Cookiecutter Data Science
# https://cookiecutter-data-science.drivendata.org/

INPUT_DATA_DIR = Path("../data/raw")
OUTPUT_DATA_DIR = Path("../data/processed")
OUTPUT_FIGURES_DIR = Path("../reports/figures")

# -----------------------------------------------------------------
# 3. Definimos algunas otras variables que usaremos más adelante
ASIGNATURAS_CHILENAS = [
    "math score",
    "reading score",
    "writing score",
    "history score",
]

ASIGNATURAS = [
    "math score",
    "reading score",
    "writing score",
    "history score",
    "science score"
]

### 1.2 — Leer y reconstruir las fuentes (0,2 puntos)

La primera tarea será cargar los registros entregados por el colegio. Antes de hacerlo, es conveniente que nos detengamos un momento a pensar en concepto fundamental y que aparecerá constantemente durante el curso y (y probablemente en sus vidas): **I/O** .

Para comprender más de esto, respondan las siguientes preguntas:

<!-- PREGUNTA -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #58a6ff; background:rgba(88,166,255,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">❓ Pregunta — API, documentación e I/O</strong>

Antes de comenzar, revisen la [documentación de Polars](https://docs.pola.rs/) y respondan brevemente:

1. ¿Qué se entiende por una **API** en el contexto de una librería como `Polars`?

2. ¿Qué diferencia existe entre la **API Reference** y una **User Guide**?

3. ¿Qué significa **I/O** (*Input/Output*) en programación y ciencia de datos?

4. ¿Qué operaciones que realizarán en este laboratorio considerarían operaciones de **I/O**? Mencionen al menos tres ejemplos.

5. Exploren la documentación de [`Polars`](https://docs.pola.rs/user-guide/io/) e identifiquen **cinco métodos o funciones que les parezcan interesantes**. Para cada uno, indiquen brevemente qué permite hacer y los formatos involucrados.

</div>

<code>Escriban su respuesta aquí:</code>

<code>Escriban su respuesta aquí:</code>

#### 1.3. Leer los fragmentos del registro principal

Ahora, toca aplicar los métodos de I/O.
El sistema principal del colegio entregó sus datos repartidos en dos archivos JSON:

- `students_grades_1.json`;
- `students_grades_2.json`.

Utilicen el lector adecuado de `Polars` para cargar **cada archivo por separado** en un `DataFrame`. Usen las rutas `INPUT_DATA_DIR / "students_grades_X.json"` (objetos `Path`, de pathlib) respectivamente para acceder a los archivos.


Por ahora, **no les hagan nada más, simplemente cárguenlos**.

In [ ]:
# Su código aquí
raise NotImplementedError(
    "Completen la lectura de los dos fragmentos JSON antes de continuar."
)

### 1.3 — Explorar los DataFrames

<!-- LORE -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #0f9e99; background:rgba(15,158,153,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">🧰 Una ayuda del equipo de TI</strong>

  Antes de entregarles los registros, el equipo de TI del colegio preparó y utilizó en su momento una función para facilitar la inspección inicial de los datos.

  Esta herramienta permite revisar rápidamente la estructura de cada fuente, detectar posibles problemas y entender mejor qué información recibió cada sistema.

  Su trabajo será utilizarla para realizar una primera auditoría antes de comenzar a combinar los registros.

</div>


Utilicen la función `explorar_tabla`, que, dado un `DataFrame`, les permitirá revisar:

- Sus dimensiones y columnas;
- El esquema y los tipos de datos;
- Las primeras y últimas filas;
- Una muestra aleatoria;
- La cantidad y porcentaje de valores nulos;
- La cantidad de valores únicos por columna;
- Y posibles claves repetidas, junto con un resumen de cuántos registros están involucrados.

In [ ]:
def explorar_tabla(
    tabla: pl.DataFrame,
    nombre: str,
    clave: str = "names",
) -> None:
    """Muestra un perfil reproducible de una tabla."""

    print("=" * 60)
    print(f"📋 {nombre}")
    print("=" * 60)

    print("\nDimensiones:")
    print(f"{tabla.height} filas × {tabla.width} columnas")

    print("\nColumnas:")
    print(tabla.columns)

    print("\nEsquema:")
    print(tabla.schema)

    print("\nPrimeras filas:")
    display(tabla.head(5))

    print("\nÚltimas filas:")
    display(tabla.tail(5))

    print("\nMuestra aleatoria:")
    display(
        tabla.sample(
            n=min(5, tabla.height),
            seed=7202,
        )
    )

    print("\nNulos por columna:")
    display(tabla.null_count())

    print("\nPorcentaje de nulos por columna:")
    display(
        tabla.select(
            (
                pl.all().null_count()
                / tabla.height
                * 100
            ).round(2)
        )
    )

    print("\nValores únicos por columna:")
    display(
        tabla.select(
            pl.all().n_unique()
        )
    )

    if clave in tabla.columns:
        repetidas = (
            tabla.group_by(clave)
            .len()
            .filter(pl.col("len") > 1)
            .sort("len", descending=True)
        )

        print(f"\nClaves repetidas en `{clave}`:")
        print(f"Cantidad de claves repetidas: {repetidas.height}")
        print(
            "Filas involucradas:",
            repetidas["len"].sum() if repetidas.height > 0 else 0,
        )

        display(repetidas)

    print("\n" + "=" * 60)

Apliquen esta función sobre las distintas fuentes y comparen sus características antes de realizar cualquier concatenación o unión.

En esta etapa, el objetivo es **entender el estado de los datos**, no corregirlos todavía. Presten especial atención a diferencias de estructura, tipos de datos, valores faltantes y posibles problemas en las claves que puedan ser relevantes en las etapas siguientes.

In [ ]:
explorar_tabla(df_student_grades_1, nombre="Student Grades 1")

In [ ]:
explorar_tabla(df_student_grades_1, nombre="Student Grades 1")

<!-- PREGUNTA -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #58a6ff; background:rgba(88,166,255,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">❓ Pregunta — Primera auditoría de los registros</strong>

A partir de los resultados entregados por `explorar_tabla` para ambos dataframes, respondan:

1. ¿Qué representa **una fila** en estos `DataFrame`? ¿Qué datos tienen? ¿Cuál parece ser la unidad de observación?

2. ¿Qué columna podría utilizarse como **identificador de un estudiante**? ¿Qué evidencia de la exploración respalda esta elección?

3. Comparen ambos `DataFrame`: ¿tienen las mismas columnas, en el mismo orden y con los mismos tipos de datos? ¿Qué les indica esto sobre la relación entre ambos archivos?

4. ¿Existen valores nulos o claves repetidas? En caso de encontrarlos, ¿por qué podrían convertirse en un problema en etapas posteriores del análisis?

5. Si ambos archivos corresponden efectivamente a fragmentos de un mismo registro, ¿cuántas filas esperarían obtener al reconstruirlo? ¿Qué condición debería cumplirse para que simplemente agregar las filas de ambos archivos tenga sentido?

6. Aunque names no presenta repeticiones en estos archivos, ¿es un identificador confiable de estudiantes en un sistema real? Expliquen qué problemas podrían aparecer y qué tipo de columna sería preferible usar como clave.

</div>

<code>Escriban su respuesta aquí:</code>

<code>Escriban su respuesta aquí:</code>

### 1.4 — Concatenación

<!-- LORE -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #0f9e99; background:rgba(15,158,153,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">🧩 Reconstruyendo el registro del colegio</strong>

  La primera revisión confirmó lo dicho por el director: los dos archivos JSON tienen la misma estructura y corresponden a fragmentos del mismo registro de estudiantes.

  El problema es que, por razones históricas del sistema, la información quedó repartida en dos archivos distintos. El Director Polar necesita volver a disponer de un único registro antes de continuar con cualquier análisis.

  Su siguiente tarea será entonces <strong>reconstruir el registro principal</strong>, combinando ambos fragmentos sin alterar sus columnas ni perder estudiantes en el proceso.
</div>


<!-- DEFINICIÓN -->

<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #3fb950; background:rgba(63,185,80,.12); color:inherit; border-radius:4px;">

<strong style="display:block; margin-bottom:8px;">📖 Definición — Concatenación de <code>DataFrame</code></strong>

La **concatenación** permite unir dos o más <code>DataFrame</code> siguiendo su posición y estructura, sin buscar correspondencias entre sus valores.

El objetivo de esta operación es **extender una tabla**: podemos agregar nuevas filas cuando las fuentes representan observaciones del mismo tipo, o agregar nuevas columnas cuando queremos colocar variables lado a lado.

En <code>Polars</code>, esta operación se realiza con <code>pl.concat</code>. El argumento <code>how</code> indica cómo se combinarán los <code>DataFrame</code>.

**Concatenación vertical**

La concatenación vertical **agrega filas**. Para utilizarla, los <code>DataFrame</code> deben representar el mismo tipo de observación y tener columnas compatibles.

```mermaid
flowchart LR
    A["<b>Tabla A</b><br/>names · math score<br/>Andrea · 5,2<br/>Benjamín · 6,1"]
    B["<b>Tabla B</b><br/>names · math score<br/>Camila · 5,8<br/>Diego · 6,4"]
    C["<b>pl.concat</b><br/>how='vertical'"]
    R["<b>Resultado</b><br/>names · math score<br/>Andrea · 5,2<br/>Benjamín · 6,1<br/>Camila · 5,8<br/>Diego · 6,4"]

    A --> C
    B --> C
    C -->|"agrega filas"| R
```

En este caso, el número de columnas se mantiene y aumenta el número de filas.

**Concatenación horizontal**

La concatenación horizontal **agrega columnas**. Las filas se emparejan según su posición, por lo que esta operación solo tiene sentido cuando existe una correspondencia conocida entre las filas de ambas tablas.

```mermaid
flowchart LR
    A["<b>Tabla A</b><br/>names<br/>Andrea<br/>Benjamín"]
    B["<b>Tabla B</b><br/>science score<br/>5,9<br/>6,4"]
    C["<b>pl.concat</b><br/>how='horizontal'"]
    R["<b>Resultado</b><br/>names · science score<br/>Andrea · 5,9<br/>Benjamín · 6,4"]

    A --> C
    B --> C
    C -->|"agrega columnas"| R
```

En este caso, el número de filas se mantiene y aumenta el número de columnas.

</div>

<!-- PREGUNTA -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #58a6ff; background:rgba(88,166,255,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">❓ Pregunta — Reconstrucción del registro</strong>

  Ahora deberán implementar la concatenación de los dos fragmentos del registro principal. Para hacerlo correctamente, primero deben decidir <strong>en qué dirección concatenar los <code>DataFrame</code></strong>.

  Respondan antes de implementar:

  1. ¿Qué tipo de concatenación corresponde utilizar en este caso: <code>vertical</code> u <code>horizontal</code>? Justifiquen su elección considerando qué representa cada fila, la estructura de ambos <code>DataFrame</code> y qué esperan obtener como resultado.

  2. Supongan ahora que, en vez de simplemente agregar filas o columnas, necesitaran combinar dos tablas buscando correspondencias entre estudiantes mediante una variable como <code>names</code>. ¿Seguiría siendo apropiado utilizar una concatenación? ¿Qué tipo de operación sería más adecuada y por qué?

</div>

<code>Escriban su respuesta aquí:</code>

<code>Escriban su respuesta aquí:</code>

Ahora, implementen ahora la concatenación de ambos `DataFrame` utilizando `pl.concat` y la orientación que justificaron anteriormente.

Guarden el resultado en `df_registro_principal` y comprueben que la tabla reconstruida tenga las dimensiones esperadas.

In [ ]:
# Su código aquí
raise NotImplementedError(
    "Completen la reconstrucción del registro antes de continuar."
)

Luego, comprueben que todo salió bien usando la función `explorar_tabla`:

In [ ]:
explorar_tabla(df_registro_principal, "Registro Principal")

### 1.5 — Leer el CSV adicional

El segundo sistema del colegio entregó las notas adicionales en el archivo `other_grades.csv`. A diferencia de los fragmentos anteriores, esta fuente utiliza formato CSV, por lo que deberán cargarla utilizando `pl.read_csv` y guardando sus resultados en `df_notas_adicionales`.

Finalmente, revisen sus dimensiones, esquema y algunas filas para comprobar que la carga se realizó correctamente.

In [ ]:
# Su código aquí
raise NotImplementedError(
    "Completen la lectura del CSV adicional antes de continuar."
)

Muestren ahora una descripción rápida de los datos adicionales usando `explorar_tabla`.

In [ ]:
explorar_tabla(df_notas_adicionales, "Notas Adicionales")

### 1.9 — Trasladar I/O al módulo (0,2 puntos)

Implementen `leer_fragmentos_json`, `leer_notas_adicionales` en `src/gradeslab/io.py`. Las funciones reciben rutas y devuelven el `DataFrame` recién guardado.

In [ ]:
# Su código aquí
raise NotImplementedError(
    "Completen las funciones de lectura antes de comparar el módulo."
)

<!-- COMPROBACIÓN -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #2da44e; background:rgba(45, 164, 78,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">🧪 Comprobación de la etapa 1</strong>
  Ejecuten <code>uv run pytest -m etapa1</code>. Deben obtener 875 filas en
  el registro principal y pasar las pruebas de tipos, rutas y roundtrip.
</div>

---
## Etapa 2 — Uniones y auditoría de claves (1,2 puntos)

### 2.1 — Auditoría antes de unir (0,2 puntos)



<!-- LORE -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #0f9e99; background:rgba(15,158,153,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">🧩 El misterio de los registros que no calzan</strong>

  El registro principal ya está reconstruido, pero el Director Polar todavía tiene un problema: el sistema adicional contiene <strong>1000 registros de notas</strong>, mientras que la matrícula oficial del colegio considera solo <strong>875 estudiantes</strong>.

  Para descubrir de dónde vienen esas diferencias, necesitarán comparar ambas fuentes y determinar <strong>qué estudiantes aparecen en las dos, cuáles existen solo en una de ellas y qué información debe conservarse</strong>.

  Esta vez no basta con colocar una tabla debajo de otra. Necesitamos relacionar sus filas utilizando una clave común, como <code>names</code>. Para eso utilizaremos distintas variantes de una de las operaciones fundamentales del trabajo con datos: los <strong><em>joins</em></strong>.

  Su tarea será probar distintas formas de unión, observar qué registros conserva cada una y decidir finalmente cuál representa mejor el registro que debería utilizar el colegio.
</div>

<!-- DEFINICIÓN -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #3fb950; background:rgba(63,185,80,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">📖 Definición — <em>joins</em>, claves y cardinalidad</strong>

  Un <strong><em>join</em></strong> permite combinar información proveniente de dos tablas buscando correspondencias entre sus filas a partir de una o más <strong>claves de unión</strong>.

  A diferencia de una concatenación, un <em>join</em> no combina filas o columnas simplemente por su posición. En cambio, compara los valores de una clave común y utiliza esas coincidencias para relacionar los registros.

  Por ejemplo, en:

  <pre><code>tabla_izquierda.join(
      tabla_derecha,
      on="names",
      how="left",
  )</code></pre>

  <code>names</code> corresponde a la clave utilizada para buscar coincidencias entre ambas tablas. Los términos <strong>izquierda</strong> y <strong>derecha</strong> indican la posición de cada tabla dentro de la operación, y el argumento <code>how</code> determina qué filas se conservarán en el resultado.

  Antes de realizar un <em>join</em>, es importante revisar la <strong>cardinalidad</strong> de la relación. La cardinalidad describe cuántas veces puede aparecer una misma clave en cada tabla.

  Por ejemplo:

  <ul>
    <li><code>1:1</code>: cada clave aparece como máximo una vez en ambas tablas;</li>
    <li><code>1:m</code>: una clave puede aparecer una vez en una tabla y varias veces en la otra;</li>
    <li><code>m:m</code>: una misma clave puede aparecer varias veces en ambas tablas.</li>
  </ul>

  Esta distinción es importante porque un <em>join</em> genera una fila por cada combinación de coincidencias. Si una clave aparece repetida inesperadamente, el resultado puede contener más filas que las tablas originales y cambiar la <strong>unidad de observación</strong> del análisis.

  Cuando esperamos una relación <code>1:1</code>, podemos utilizar <code>validate="1:1"</code> para comprobar este supuesto y evitar aceptar silenciosamente una unión con claves duplicadas.

</div>

Para encontrar estas diferencias, utilizaremos distintos tipos de `joins`, vistos desde un punto de vista de conjuntos:

### 2.2 — Semi join (0,1 puntos)


<!-- PREGUNTA -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #58a6ff; background:rgba(88,166,255,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">❓ Pregunta — Semi join</strong>

  Supongan que ejecutamos un <code>semi join</code> desde <code>df_registro_principal</code> hacia <code>df_notas_adicionales</code>, utilizando <code>names</code> como clave.

  Sean:

  - \(A\): el conjunto de estudiantes presentes en `df_registro_principal`;
  - \(B\): el conjunto de estudiantes presentes en `df_notas_adicionales`.


  1. Pensando en los conjuntos \(A\) y \(B\), ¿qué conjunto de estudiantes debería conservar esta operación?


  > **Pista:** pueden apoyarse en la [documentación oficial de `DataFrame.join` de Polars](https://docs.pola.rs/api/python/stable/reference/dataframe/api/polars.DataFrame.join.html) para revisar el comportamiento de `how="semi"` y de los demás tipos de `join`.

</div>

<code>Escriban su respuesta aquí:</code>

<code>Escriban su respuesta aquí:</code>

Ahora implementen el `semi join` utilizando `names` como clave y `df_registro_principal` como tabla izquierda.

Guarden el resultado en `registro_semi`.

Luego:

- comprueben sus dimensiones;
- muestren algunas filas;
- y verifiquen si todos los estudiantes del registro principal tienen una correspondencia en `df_notas_adicionales`.

In [ ]:
# Su código aquí
raise NotImplementedError("Completen el semi join antes de continuar.")

Observen sus dimensiones y algunas claves.

In [ ]:
print("Semi join:", registro_semi.shape)
display(registro_semi.select("names").head(5))

### 2.3 — Full join u outer join (0,1 puntos)

<!-- PREGUNTA -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #58a6ff; background:rgba(88,166,255,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">❓ Pregunta — Full join</strong>

  Supongan que ejecutamos un <code>full join</code> entre <code>df_registro_principal</code> y <code>df_notas_adicionales</code>, utilizando <code>names</code> como clave.

  Sean:

  - \(A\): el conjunto de estudiantes presentes en `df_registro_principal`;
  - \(B\): el conjunto de estudiantes presentes en `df_notas_adicionales`.

  1. Pensando en los conjuntos \(A\) y \(B\), ¿qué conjunto de estudiantes debería conservar esta operación?

  > **Pista:** pueden apoyarse en la [documentación oficial de `DataFrame.join` de Polars](https://docs.pola.rs/api/python/stable/reference/dataframe/api/polars.DataFrame.join.html) para revisar el comportamiento de `how="full"`.

</div>

<code>Escriban su respuesta aquí:</code>

<code>Escriban su respuesta aquí:</code>

In [ ]:
# Su código aquí
raise NotImplementedError("Completen el full join antes de continuar.")

Muestren las dimensiones y algunas filas que tengan valores nulos.

In [ ]:
print("Full join:", registro_full.shape)
display(registro_full.filter(pl.any_horizontal(pl.all().is_null())).head(5))

### 2.4 — Left join (0,1 puntos)

<!-- PREGUNTA -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #58a6ff; background:rgba(88,166,255,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">❓ Pregunta — Left join</strong>

  Supongan que ejecutamos un <code>left join</code> desde <code>df_registro_principal</code> hacia <code>df_notas_adicionales</code>, utilizando <code>names</code> como clave.

  Sean:

  - \(A\): el conjunto de estudiantes presentes en `df_registro_principal`;
  - \(B\): el conjunto de estudiantes presentes en `df_notas_adicionales`.

  1. Pensando en los conjuntos \(A\) y \(B\), ¿qué conjunto de estudiantes debería conservar esta operación?

  > **Pista:** pueden apoyarse en la [documentación oficial de `DataFrame.join` de Polars](https://docs.pola.rs/api/python/stable/reference/dataframe/api/polars.DataFrame.join.html) para revisar el comportamiento de `how="left"`.

</div>

<code>Escriban su respuesta aquí:</code>

<code>Escriban su respuesta aquí:</code>

In [ ]:
# Su código aquí
raise NotImplementedError("Completen el left join antes de continuar.")

Comprueben las dimensiones y las columnas adicionales.

In [ ]:
print("Left join:", registro_left.shape)
print("Últimas columnas:", registro_left.columns[-2:])
display(registro_left.select("names", "science score", "history score").head())

### 2.5 — Right join (0,1 puntos)

<!-- PREGUNTA -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #58a6ff; background:rgba(88,166,255,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">❓ Pregunta — Right join</strong>

  Supongan que ejecutamos un <code>right join</code> desde <code>df_registro_principal</code> hacia <code>df_notas_adicionales</code>, utilizando <code>names</code> como clave.

  Sean:

  - \(A\): el conjunto de estudiantes presentes en `df_registro_principal`;
  - \(B\): el conjunto de estudiantes presentes en `df_notas_adicionales`.

  1. Pensando en los conjuntos \(A\) y \(B\), ¿qué conjunto de estudiantes debería conservar esta operación?

  > **Pista:** pueden apoyarse en la [documentación oficial de `DataFrame.join` de Polars](https://docs.pola.rs/api/python/stable/reference/dataframe/api/polars.DataFrame.join.html) para revisar el comportamiento de `how="right"`.

</div>

<code>Escriban su respuesta aquí:</code>

<code>Escriban su respuesta aquí:</code>

In [ ]:
# Su código aquí
raise NotImplementedError("Completen el right join antes de continuar.")

Muestren las dimensiones y algunas claves que no pertenecen al registro
principal.

In [ ]:
claves_principales = df_registro_principal.select("names")
print("Right join:", registro_right.shape)
display(
    registro_right.join(
        claves_principales,
        on="names",
        how="anti",
    ).select("names").head(5)
)

### 2.6 — Inner join (0,1 puntos)

<!-- PREGUNTA -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #58a6ff; background:rgba(88,166,255,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">❓ Pregunta — Inner join</strong>

  Supongan que ejecutamos un <code>inner join</code> entre <code>df_registro_principal</code> y <code>df_notas_adicionales</code>, utilizando <code>names</code> como clave.

  Sean:

  - \(A\): el conjunto de estudiantes presentes en `df_registro_principal`;
  - \(B\): el conjunto de estudiantes presentes en `df_notas_adicionales`.

  1. Pensando en los conjuntos \(A\) y \(B\), ¿qué conjunto de estudiantes debería conservar esta operación?

  > **Pista:** pueden apoyarse en la [documentación oficial de `DataFrame.join` de Polars](https://docs.pola.rs/api/python/stable/reference/dataframe/api/polars.DataFrame.join.html) para revisar el comportamiento de `how="inner"`.

</div>

<code>Escriban su respuesta aquí:</code>

<code>Escriban su respuesta aquí:</code>

In [ ]:
# Su código aquí
raise NotImplementedError("Completen el inner join antes de continuar.")

Muestren sus dimensiones y compárenlas con las de la matrícula principal.

In [ ]:
print("Inner join:", registro_inner.shape)
print(
    "Conserva todas las claves principales:",
    registro_inner["names"].n_unique()
    == df_registro_principal["names"].n_unique(),
)

### 2.7 — Anti join (0,1 puntos)

<!-- PREGUNTA -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #58a6ff; background:rgba(88,166,255,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">❓ Pregunta — Anti join</strong>

  Supongan que ejecutamos un <code>anti join</code> desde <code>df_notas_adicionales</code> hacia <code>df_registro_principal</code>, utilizando <code>names</code> como clave.

  Sean:

  - \(A\): el conjunto de estudiantes presentes en `df_registro_principal`;
  - \(B\): el conjunto de estudiantes presentes en `df_notas_adicionales`.

  1. Pensando en los conjuntos \(A\) y \(B\), ¿qué conjunto de estudiantes debería conservar esta operación?

  > **Pista:** pueden apoyarse en la [documentación oficial de `DataFrame.join` de Polars](https://docs.pola.rs/api/python/stable/reference/dataframe/api/polars.DataFrame.join.html) para revisar el comportamiento de `how="anti"`.

</div>

<code>Escriban su respuesta aquí:</code>

<code>Escriban su respuesta aquí:</code>

In [ ]:
# Su código aquí
raise NotImplementedError("Completen la unión de producción antes de continuar.")

Muestren ambos resultados y comprueben que el anti join deja exactamente
125 registros.

In [ ]:
print("Anti join:", registros_extra.shape)
print("Registro unido:", registro_unido.shape)
display(registros_extra.head(5))
display(registro_unido.select("names", "science score", "history score").head())

### 2.8 — Interpretar y elegir la producción (0,2 puntos)

Las siguientes preguntas forman parte de esta decisión: deben justificar el
universo de análisis, los registros excluidos y la cardinalidad elegida.

<!-- PREGUNTA -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #58a6ff; background:rgba(88, 166, 255,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">❓ Pregunta 2 — Interpretar las uniones</strong>
  Comparen las dimensiones de las seis uniones. ¿Qué significan los 125 registros del anti join? ¿Qué riesgo evita validate="1:1"?
</div>

<code>Escriban su respuesta aquí:</code>

<code>Escriban su respuesta aquí:</code>

<!-- PREGUNTA -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #58a6ff; background:rgba(88, 166, 255,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">❓ Pregunta 3 — Elegir la producción</strong>
  Justifiquen por qué la tabla de producción parte desde la matrícula principal y usa left join. ¿Qué problema podría ocultar un inner join?
</div>

<code>Escriban su respuesta aquí:</code>

<code>Escriban su respuesta aquí:</code>

### 2.9 — Trasladar la unión principal al módulo (0,2 puntos)

Hasta ahora utilizaron distintos tipos de `join` para explorar la relación entre ambas fuentes. En esta etapa deberán llevar al módulo solo las operaciones que necesitaremos conservar como parte del flujo principal del proyecto.

Implementen las siguientes funciones en `src/gradeslab/joins.py`:

- `unir_registro(principal, adicional)`: debe agregar las columnas de `adicional` al registro `principal` utilizando `names` como clave. La función debe:
  - conservar todos los estudiantes del registro principal;
  - utilizar un `left join`;
  - validar que la relación entre ambas tablas sea `1:1` mediante `validate="1:1"`;
  - devolver el `DataFrame` resultante.

- `claves_solo_en_adicional(principal, adicional)`: debe identificar los registros presentes en la fuente adicional que no tienen correspondencia en el registro principal. La función debe:
  - comparar ambas tablas utilizando `names` como clave;
  - utilizar un `anti join` desde la tabla adicional;
  - devolver las filas completas de `adicional` que no aparecen en `principal`.

Las demás variantes de `join` utilizadas durante la exploración pueden permanecer únicamente en el notebook. El módulo debe contener las operaciones que representan la decisión final de integración y auditoría de las fuentes.

In [ ]:
# Su código aquí
raise NotImplementedError("Completen las funciones de unión antes de continuar.")

Comparen las dimensiones obtenidas desde el módulo.

In [ ]:
print("Unión modular:", registro_unido_modulo.shape)
print("Extras modulares:", registros_extra_modulo.shape)
print("Unión coincide:", registro_unido_modulo.equals(registro_unido))
print("Extras coinciden:", registros_extra_modulo.equals(registros_extra))

<!-- COMPROBACIÓN -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #2da44e; background:rgba(45, 164, 78,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">🧪 Comprobación de la etapa 2</strong>
  Ejecuten <code>uv run pytest -m etapa2</code>. Comprueben las dimensiones, los registros exclusivos y el fallo ante claves repetidas.
</div>

---
## Etapa 3 — Formato ancho y formato largo (1,1 puntos)

<!-- LORE -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #0f9e99; background:rgba(15,158,153,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">📚 Una tabla para cada problema</strong>

Con los registros finalmente integrados, el Director Polar quiere comenzar a comparar el rendimiento entre asignaturas.

Sin embargo, el equipo descubre un nuevo inconveniente: actualmente cada asignatura ocupa una columna distinta. Esta representación es cómoda para consultar las notas de un estudiante, pero puede resultar poco práctica cuando queremos analizar todas las asignaturas de la misma manera.

El equipo de TI les comenta que una misma información puede organizarse de distintas formas dependiendo del análisis que queramos realizar. En particular, será útil aprender a transformar los datos entre <strong>formato ancho</strong> y <strong>formato largo</strong>.

Su siguiente tarea será explorar ambas representaciones, comprender qué información conserva cada una y aprender a transformar una en la otra sin perder datos en el camino.

</div>

<!-- DEFINICIÓN -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #3fb950; background:rgba(63,185,80,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">📖 Definición — Formato ancho y formato largo</strong>

Una misma información puede representarse de distintas formas dependiendo del análisis que queramos realizar.

En una tabla en **formato ancho**, distintas mediciones de una misma unidad de observación se almacenan en columnas separadas. Por ejemplo, un estudiante puede ocupar una sola fila y tener una columna distinta para cada asignatura.

**Formato ancho**

| estudiante | matemática | lectura |
|---|---:|---:|
| A | 6.0 | 5.8 |
| B | 5.5 | 6.2 |

En una tabla en **formato largo**, esas mediciones se reorganizan en filas. Los nombres de las variables pasan a formar parte de una columna y sus valores se almacenan en otra. Como consecuencia, una misma unidad de observación puede aparecer en varias filas.

**Formato largo**

| estudiante | asignatura | puntaje |
|---|---|---:|
| A | matemática | 6.0 |
| A | lectura | 5.8 |
| B | matemática | 5.5 |
| B | lectura | 6.2 |

El formato ancho suele ser cómodo para observar varias características de una misma entidad en una sola fila. El formato largo, en cambio, facilita operaciones en las que queremos tratar distintas mediciones de manera uniforme, por ejemplo, agrupar, resumir o visualizar resultados por asignatura.

En `Polars`, `unpivot` transforma una tabla desde formato ancho hacia formato largo:

- `index` indica las columnas que identifican o describen la unidad de observación y que deben conservarse;
- `on` indica las columnas cuyos nombres y valores pasarán a distribuirse en filas.

La operación inversa se realiza con `pivot`, que utiliza los valores de una columna para volver a crear columnas.

**Importante:** para reconstruir una tabla ancha sin ambigüedad, cada combinación entre las columnas de identificación y la variable que será convertida en columna debe corresponder a un único valor. Si existen varias observaciones para una misma combinación, será necesario indicar cómo agregarlas.

</div>

### 3.1 — Un ejemplo mínimo: de ancho a largo y de vuelta (0,1 puntos)

Antes de transformar los registros reales del colegio, trabajaremos con un ejemplo pequeño en el que podamos seguir cada observación.

La siguiente tabla ya está preparada para ustedes:

In [ ]:
ejemplo_ancho = pl.DataFrame(
    {
        "estudiante": ["A", "B"],
        "matematica": [6.0, 5.5],
        "lectura": [5.8, 6.2],
    }
)

display(ejemplo_ancho)

Utilicen `unpivot` para transformar `ejemplo_ancho` a formato largo:

- mantengan `estudiante` como columna de identificación mediante `index`;
- transformen `matematica` y `lectura` mediante `on`;
- renombren `variable` como `asignatura`;
- renombren `value` como `puntaje`.

Guarden el resultado en `ejemplo_largo`.

Luego utilicen `pivot` para reconstruir la tabla original y guarden el resultado en `ejemplo_reconstruido`.

In [ ]:
# Su código aquí
raise NotImplementedError("Completen el ejemplo de formatos antes de continuar.")

<!-- PREGUNTA -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #58a6ff; background:rgba(88,166,255,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">❓ Pregunta — ¿Qué cambió?</strong>

1. ¿Qué representa una fila en `ejemplo_ancho` y qué representa una fila en `ejemplo_largo`?
2. ¿Por qué aumenta la cantidad de filas al pasar a formato largo?
3. ¿Se perdió alguna de las notas originales durante la transformación?

</div>

<code>Escriban su respuesta aquí:</code>

<code>Escriban su respuesta aquí:</code>

### 3.2 — Identificar las columnas de identidad

Ahora aplicaremos la misma transformación al registro del colegio.

Antes de utilizar `unpivot`, debemos separar las columnas según el papel que cumplen:

- las columnas de **asignaturas** contienen las mediciones que queremos llevar a filas;
- las demás columnas describen al estudiante y deben acompañar cada una de esas mediciones.

<!-- PREGUNTA -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #58a6ff; background:rgba(88,166,255,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">❓ Pregunta — Identidad y mediciones</strong>

1. ¿Cuáles columnas deberían transformarse en valores de la futura columna `asignatura`?
2. ¿Qué información debe permanecer como identificación o descripción del estudiante?

</div>

<code>Escriban su respuesta aquí:</code>

<code>Escriban su respuesta aquí:</code>

Utilicen `ASIGNATURAS` para identificar las cinco columnas que serán transformadas.

Obtengan las columnas restantes y guárdenlas en `columnas_identidad`. No es necesario utilizar ciclos para hacerlo.

In [ ]:
# Su código aquí
raise NotImplementedError("Completen las columnas de identidad antes de continuar.")

### 3.3 — Transformar el registro a formato largo (0,2 puntos)

El registro contiene **875 estudiantes** y queremos representar **5 asignaturas** en formato largo.

Antes de escribir código, respondan:

<!-- PREGUNTA -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #58a6ff; background:rgba(88,166,255,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">❓ Pregunta — Dimensiones esperadas</strong>

1. Si cada estudiante tiene una observación por cada una de las cinco asignaturas, ¿cuántas filas debería tener la tabla en formato largo?
2. ¿Qué representará ahora una fila de esta tabla?

</div>

<code>Escriban su respuesta aquí:</code>

<code>Escriban su respuesta aquí:</code>

Utilicen `unpivot` sobre `registro_unido`:

- transformen las columnas indicadas en `ASIGNATURAS`;
- mantengan `columnas_identidad` mediante `index`;
- renombren `variable` como `asignatura`;
- renombren `value` como `puntaje`.

Guarden el resultado en `notas_largas`.

In [ ]:
# Su código aquí
raise NotImplementedError("Completen la tabla larga antes de continuar.")

Comprueben que las dimensiones obtenidas coincidan con lo esperado y observen algunas filas.

Verifiquen además que cada estudiante aparezca exactamente cinco veces, una por asignatura.

In [ ]:
print("Tabla larga:", notas_largas.shape)
display(notas_largas.head(10))

display(
    notas_largas.group_by("names")
    .len()
    .select(
        pl.col("len").min().alias("mínimo"),
        pl.col("len").max().alias("máximo"),
    )
)

### 3.4 — Conservar la escala de cada puntaje (0,2 puntos)

La transformación anterior reunió las cinco asignaturas en una única columna llamada `puntaje`.

Sin embargo, hay una diferencia importante entre ellas: `science score` está expresada como porcentaje, mientras que las otras cuatro asignaturas utilizan la escala chilena de notas.

Esto significa que dos valores presentes en `puntaje` no necesariamente utilizan la misma unidad.

<!-- PREGUNTA -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #58a6ff; background:rgba(88,166,255,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">❓ Pregunta — ¿Qué significa un puntaje?</strong>

1. ¿Sería correcto comparar o promediar directamente `science score` con las demás asignaturas?
2. ¿Qué información deberíamos conservar para interpretar correctamente cada valor de `puntaje`?

</div>

<code>Escriban su respuesta aquí:</code>

<code>Escriban su respuesta aquí:</code>

Agreguen una columna `escala` utilizando expresiones de `Polars`:

- para `science score`, asignen `"porcentaje"`;
- para las demás asignaturas, asignen `"escala chilena"`.

Luego comprueben qué asignaturas pertenecen a cada escala.

In [ ]:
# Su código aquí
raise NotImplementedError("Completen la columna escala antes de continuar.")

In [ ]:
display(
    notas_largas.group_by("escala", "asignatura")
    .len()
    .sort("escala", "asignatura")
)

### 3.5 — Auditar la unicidad antes de reconstruir (0,1 puntos)

Queremos comprobar que el formato largo puede convertirse nuevamente en formato ancho **sin tener que decidir arbitrariamente entre varios valores**.

Para esto, cada estudiante debe tener como máximo un puntaje para cada asignatura.

<!-- PREGUNTA -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #58a6ff; background:rgba(88,166,255,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">❓ Pregunta — Clave de la tabla larga</strong>

1. ¿Qué combinación de columnas debería identificar de manera única una nota en `notas_largas`?
2. ¿Qué problema aparecería al ejecutar `pivot` si esa combinación tuviera más de un puntaje?

</div>

<code>Escriban su respuesta aquí:</code>

<code>Escriban su respuesta aquí:</code>

Agrupen por `names` y `asignatura`, cuenten las observaciones y conserven solamente aquellas combinaciones que aparezcan más de una vez.

Guarden el resultado en `combinaciones_repetidas`.

La tabla resultante debería estar vacía.

In [ ]:
# Su código aquí
raise NotImplementedError("Completen la comprobación de unicidad antes de continuar.")

### 3.6 — Reconstruir el registro con `pivot` (0,2 puntos)

La auditoría anterior confirmó que cada combinación estudiante-asignatura contiene un único puntaje. Por lo tanto, podemos intentar el viaje de regreso: convertir `notas_largas` nuevamente a formato ancho.

Utilicen `pivot`:

- `asignatura` debe determinar las nuevas columnas;
- `columnas_identidad` debe identificar las filas;
- `puntaje` debe proporcionar los valores.

Guarden el resultado en `registro_reconstruido`.

**No utilicen una función de agregación:** si existieran múltiples valores para una misma combinación, preferimos que la operación falle en vez de elegir uno silenciosamente.

In [ ]:
# Su código aquí
raise NotImplementedError("Completen el pivot antes de continuar.")

Comprueben que:

- existen nuevamente 875 filas;
- las cinco asignaturas volvieron a ser columnas;
- cada estudiante aparece una sola vez;
- las columnas de identidad siguen presentes.

In [ ]:
print("Tabla reconstruida:", registro_reconstruido.shape)
print("Columnas finales:", registro_reconstruido.columns)
print("Estudiantes únicos:", registro_reconstruido["names"].n_unique())
display(registro_reconstruido.head(5))

<!-- PREGUNTA -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #58a6ff; background:rgba(88,166,255,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">❓ Pregunta — Elegir una representación</strong>

1. Si el Director Polar quisiera comparar el rendimiento de las distintas asignaturas mediante una única operación de `group_by`, ¿qué formato sería más conveniente? ¿Por qué?
2. Si quisiera consultar en una sola fila todas las notas de un estudiante, ¿qué formato sería más cómodo?
3. ¿Cambió la información académica al pasar de ancho a largo y nuevamente a ancho, o solamente su representación?

</div>

<code>Escriban su respuesta aquí:</code>

<code>Escriban su respuesta aquí:</code>

### 3.7 — Llevar la reestructuración al módulo (0,3 puntos)

Durante esta etapa construyeron paso a paso el flujo necesario para cambiar la representación de las notas. Ahora deberán encapsular ese comportamiento en funciones reutilizables.

Implementen en `src/gradeslab/reshaping.py`:

#### `pasar_a_formato_largo(registro)`

Debe recibir el registro integrado en formato ancho y devolver su representación en formato largo.

La función debe:

- considerar como asignaturas las columnas definidas en `ASIGNATURAS`;
- conservar las demás columnas como información de identidad;
- transformar las asignaturas mediante `unpivot`;
- producir las columnas `asignatura` y `puntaje`;
- agregar la columna `escala`, distinguiendo `science score` de las asignaturas en escala chilena.

Para los datos del laboratorio, el resultado debe contener **4375 filas**.

#### `pasar_a_formato_ancho(notas_largas)`

Debe recibir la tabla larga y reconstruir el formato ancho mediante `pivot`.

La función debe:

- utilizar `asignatura` para reconstruir las cinco columnas de notas;
- utilizar `puntaje` como valor;
- conservar las columnas de identidad del estudiante;
- no utilizar una agregación que oculte posibles combinaciones duplicadas;
- devolver una tabla con **875 filas**.

Las funciones deben reproducir las transformaciones que desarrollaron previamente en el notebook.

In [ ]:
# Su código aquí
raise NotImplementedError("Completen las funciones de reestructuración antes de continuar.")

Comparen las dimensiones y los resultados obtenidos desde el módulo con los construidos anteriormente en el notebook.

In [ ]:
print("Largo modular:", notas_largas_modulo.shape)
print("Ancho modular:", registro_reconstruido_modulo.shape)
print("Largo coincide:", notas_largas_modulo.equals(notas_largas))
print(
    "Ancho coincide:",
    registro_reconstruido_modulo.equals(registro_reconstruido),
)

<!-- COMPROBACIÓN -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #2da44e; background:rgba(45,164,78,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">🧪 Comprobación de la etapa 3</strong>

Antes de continuar, ejecuten <code>uv run pytest -m etapa3</code>.

Comprueben que:

- la tabla larga contiene 4375 observaciones;
- cada estudiante tiene cinco observaciones, una por asignatura;
- las escalas quedan correctamente identificadas;
- no existen combinaciones repetidas de estudiante y asignatura;
- el <code>pivot</code> reconstruye los 875 estudiantes;
- y las funciones del módulo reproducen los resultados obtenidos en el notebook.

</div>